PENGUMPULAN DATASET



```
TAHAP 1: Dataset benchmark standar CVRPLIB
```



In [ ]:
pip install vrplib

In [ ]:
import numpy as np
import vrplib

def load_cvrp_instance(name, folder="data/cvrplib"):
    # Membaca isi file .vrp
    inst = vrplib.read_instance(f"{folder}/{name}.vrp")

    coords = inst["node_coord"]
    # Hitung matriks jarak Euclidean
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    dist_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))

    # Terapkan konvensi pembulatan CVRPLIB
    if name.startswith("A-") or name.startswith("X-"):
        dist_matrix = np.round(dist_matrix)
    # Untuk CMT*, biarkan tanpa pembulatan

    inst["edge_weight"] = dist_matrix
    return inst

# Contoh penggunaan & membaca nilai optimal jarak dari file .sol
data_a32 = load_cvrp_instance("A-n32-k5")
sol_a32 = vrplib.read_solution("data/cvrplib/A-n32-k5.sol")

print("Kapasitas kendaraan (Q):", data_a32["capacity"])
print("Permintaan per node:", data_a32["demand"])
print("Nilai optimal jarak (.sol):", sol_a32["cost"])

Kapasitas kendaraan (Q): 100
Permintaan per node: [ 0 19 21  6 19  7 12 16  6 16  8 14 21 16  3 22 18 19  1 24  8 12  4  8
 24 24  2 20 15  2 14  9]
Nilai optimal jarak (.sol): 784


In [ ]:
import os
import pandas as pd

os.makedirs("data", exist_ok=True)

bks_data = [
    {"instance": "A-n32-k5",   "objektif": "jarak", "nilai": 784.0,    "sumber": "Uchoa dkk. (2017) Tabel A.9"},
    {"instance": "A-n45-k7",   "objektif": "jarak", "nilai": 1146.0,   "sumber": "Uchoa dkk. (2017) Tabel A.9"},
    {"instance": "A-n60-k9",   "objektif": "jarak", "nilai": 1354.0,   "sumber": "Uchoa dkk. (2017) Tabel A.9"},
    {"instance": "A-n80-k10",  "objektif": "jarak", "nilai": 1763.0,   "sumber": "Uchoa dkk. (2017) Tabel A.9"},
    {"instance": "CMT1",       "objektif": "FCVRP", "nilai": 751.11,   "sumber": "Kramer dkk. (2014) Tabel 2"},
    {"instance": "CMT2",       "objektif": "FCVRP", "nilai": 1172.62,  "sumber": "Kramer dkk. (2014) Tabel 2"},
    {"instance": "CMT3",       "objektif": "FCVRP", "nilai": 1147.83,  "sumber": "Kramer dkk. (2014) Tabel 2"},
    {"instance": "CMT12",      "objektif": "FCVRP", "nilai": 1174.02,  "sumber": "Kramer dkk. (2014) Tabel 2"},
    {"instance": "X-n101-k25", "objektif": "jarak", "nilai": 27591.0,  "sumber": "Uchoa dkk. (2017) Tabel 1"},
    {"instance": "X-n110-k13", "objektif": "jarak", "nilai": 14971.0,  "sumber": "Uchoa dkk. (2017) Tabel 1"},
    {"instance": "X-n120-k6",  "objektif": "jarak", "nilai": 13332.0,  "sumber": "Uchoa dkk. (2017) Tabel 1"},
]

df_bks = pd.DataFrame(bks_data)
df_bks.to_csv("data/bks.csv", index=False)

# Tampilkan isi tabel untuk verifikasi
display(df_bks)

,instance,objektif,nilai,sumber
0,A-n32-k5,jarak,784.00,Uchoa dkk. (2017) Tabel A.9
1,A-n45-k7,jarak,1146.00,Uchoa dkk. (2017) Tabel A.9
2,A-n60-k9,jarak,1354.00,Uchoa dkk. (2017) Tabel A.9
3,A-n80-k10,jarak,1763.00,Uchoa dkk. (2017) Tabel A.9
4,CMT1,FCVRP,751.11,Kramer dkk. (2014) Tabel 2
5,CMT2,FCVRP,1172.62,Kramer dkk. (2014) Tabel 2
6,CMT3,FCVRP,1147.83,Kramer dkk. (2014) Tabel 2
7,CMT12,FCVRP,1174.02,Kramer dkk. (2014) Tabel 2
8,X-n101-k25,jarak,27591.00,Uchoa dkk. (2017) Tabel 1
9,X-n110-k13,jarak,14971.00,Uchoa dkk. (2017) Tabel 1


Tahap 2


In [4]:
import os
import numpy as np
import geopandas as gpd
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

os.makedirs("data/semarang", exist_ok=True)

# 1. Baca file GeoJSON hasil ekspor Overpass Turbo
gdf = gpd.read_file("data/semarang/export.geojson")

# 2. Proyeksi ke UTM 49S (EPSG:32749) untuk centroid akurat tanpa UserWarning
gdf_utm = gdf.to_crs(epsg=32749)
centroids_utm = gdf_utm.geometry.centroid
centroids_wgs84 = centroids_utm.to_crs(epsg=4326)

gdf["lon"] = centroids_wgs84.x
gdf["lat"] = centroids_wgs84.y
gdf["x_m"] = centroids_utm.x
gdf["y_m"] = centroids_utm.y

# Bersihkan kolom nama
if "name" not in gdf.columns:
    gdf["name"] = [f"Titik Logistik {i+1}" for i in range(len(gdf))]
else:
    gdf["name"] = gdf["name"].fillna("Tanpa Nama")

# 3. Filter Batas Geografis Kota Semarang & Jarak Minimum Antar-Titik
# Saring koordinat di dalam batas wilayah administratif Kota Semarang
gdf = gdf[
    (gdf["lat"] >= -7.12) & (gdf["lat"] <= -6.93) &
    (gdf["lon"] >= 110.27) & (gdf["lon"] <= 110.50)
].copy()

# Hapus duplikat nama
gdf = gdf.drop_duplicates(subset=["name"]).reset_index(drop=True)

# Hapus titik yang berjarak < 300 meter satu sama lain (mencegah duplikasi kompleks pasar yang sama)
keep_indices = []
coords_m = gdf[["x_m", "y_m"]].values
for i in range(len(gdf)):
    if not keep_indices:
        keep_indices.append(i)
    else:
        dists = np.linalg.norm(coords_m[keep_indices] - coords_m[i], axis=1)
        if np.min(dists) >= 300.0:  # minimal jarak 300 meter
            keep_indices.append(i)

gdf = gdf.iloc[keep_indices].copy()

# 4. Deteksi Kecamatan & Validasi Terhadap Daftar 16 Kecamatan Resmi Kota Semarang
KECAMATAN_SEMARANG = {
    "Banyumanik", "Candisari", "Gajahmungkur", "Gayamsari",
    "Genuk", "Gunungpati", "Mijen", "Ngaliyan",
    "Pedurungan", "Semarang Barat", "Semarang Selatan", "Semarang Tengah",
    "Semarang Timur", "Semarang Utara", "Tembalang", "Tugu"
}

print(f"Mendeteksi kecamatan untuk {len(gdf)} kandidat titik bersih...")
geolocator = Nominatim(user_agent="green_cvrp_semarang_v2")
geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def get_valid_kecamatan(row):
    try:
        loc = geocode((row["lat"], row["lon"]), language="id")
        if loc and "address" in loc.raw:
            addr = loc.raw["address"]
            # Periksa seluruh field alamat untuk mencocokkan nama 16 kecamatan resmi
            for val in addr.values():
                clean_val = str(val).replace("Kecamatan ", "").strip()
                if clean_val in KECAMATAN_SEMARANG:
                    return clean_val
    except Exception:
        pass
    return "Tidak Valid"

gdf["kecamatan"] = gdf.apply(get_valid_kecamatan, axis=1)

# Buang titik yang tidak terdeteksi masuk ke 16 kecamatan Kota Semarang
gdf_valid = gdf[gdf["kecamatan"] != "Tidak Valid"].copy()

# 5. Pilih maksimal 30 titik secara merata lintas kecamatan (Round-Robin per kecamatan)
selected_rows = []
grouped = {kec: group.index.tolist() for kec, group in gdf_valid.groupby("kecamatan")}

while len(selected_rows) < 30 and any(grouped.values()):
    for kec in sorted(grouped.keys()):
        if grouped[kec] and len(selected_rows) < 30:
            selected_rows.append(grouped[kec].pop(0))

gdf_final = gdf_valid.loc[selected_rows].reset_index(drop=True)

# 6. Susun tabel akhir (Depot id=0 + Pelanggan id=1..n)
df_pelanggan = pd.DataFrame({
    "id": range(1, len(gdf_final) + 1),
    "nama": gdf_final["name"].values,
    "lat": gdf_final["lat"].values,
    "lon": gdf_final["lon"].values,
    "kecamatan": gdf_final["kecamatan"].values
})

df_depot = pd.DataFrame([{
    "id": 0,
    "nama": "Depot Logistik Ronggowarsito",
    "lat": -6.96345,
    "lon": 110.42785,
    "kecamatan": "Semarang Utara"
}])

df_titik = pd.concat([df_depot, df_pelanggan], ignore_index=True)
df_titik.to_csv("data/semarang/titik.csv", index=False)

print(f"\nSelesai! Total titik tersimpan: {len(df_titik)} (1 Depot + {len(df_pelanggan)} Pelanggan)")
print("\nSebaran titik per kecamatan:")
print(df_titik["kecamatan"].value_counts())

Mendeteksi kecamatan untuk 54 kandidat titik bersih...

Selesai! Total titik tersimpan: 31 (1 Depot + 30 Pelanggan)

Sebaran titik per kecamatan:
kecamatan
Semarang Utara      3
Banyumanik          3
Candisari           3
Gayamsari           3
Genuk               2
Mijen               2
Semarang Selatan    2
Ngaliyan            2
Pedurungan          2
Semarang Barat      2
Semarang Timur      2
Semarang Tengah     2
Tembalang           2
Gajahmungkur        1
Name: count, dtype: int64


In [9]:
import pandas as pd

# 1. Baca file sambil melewati 4 baris header bertingkat
df = pd.read_csv(
    "data/semarang/Jumlah Penduduk Menurut Kecamatan dan Jenis Kelamin, 2025.csv",
    skiprows=4,
    names=["kecamatan", "laki_laki", "perempuan"],
)

# 2. Hapus baris total 'Kota Semarang' dan bersihkan spasi berlebih
df["kecamatan"] = df["kecamatan"].str.strip()
df = df[df["kecamatan"] != "Kota Semarang"].copy()

# 3. Jumlahkan Laki-laki + Perempuan menjadi kolom 'penduduk'
df["penduduk"] = df["laki_laki"] + df["perempuan"]

# 4. (Opsional) Samakan penulisan spasi dengan standar batas administrasi OSM/GeoPandas
mapping_nama = {
    "Gunung Pati": "Gunungpati",
    "Gajah Mungkur": "Gajahmungkur",
}
df["kecamatan"] = df["kecamatan"].replace(mapping_nama)

# 5. Simpan hasil akhir
df[["kecamatan", "penduduk"]].to_csv(
    "data/semarang/penduduk_kecamatan.csv", index=False
)
print(df[["kecamatan", "penduduk"]])

           kecamatan  penduduk
0              Mijen     96357
1         Gunungpati    102429
2         Banyumanik    144087
3       Gajahmungkur     56328
4   Semarang Selatan     61981
5          Candisari     75396
6          Tembalang    204865
7         Pedurungan    198452
8              Genuk    141033
9          Gayamsari     70381
10    Semarang Timur     66482
11    Semarang Utara    117865
12   Semarang Tengah     55213
13    Semarang Barat    149357
14              Tugu     34398
15          Ngaliyan    147797


In [10]:
import numpy as np
import pandas as pd

# Baca data titik dan populasi kecamatan
df_titik = pd.read_csv("data/semarang/titik.csv")
df_penduduk = pd.read_csv("data/semarang/penduduk_kecamatan.csv")

# Gabungkan data berdasarkan nama kecamatan
df = df_titik.merge(df_penduduk, on="kecamatan", how="left")

# Tetapkan seed acak agar hasil konsisten
np.random.seed(42)

# Normalisasi bobot penduduk (0 s.d. 1)
min_pop = df["penduduk"].min()
max_pop = df["penduduk"].max()
bobot = (df["penduduk"] - min_pop) / (max_pop - min_pop)

# Bangkitkan demand 100 - 800 kg yang dipengaruhi bobot kecamatan + variasi acak
base_demand = 100 + (bobot * 500) + np.random.randint(0, 200, size=len(df))
df["demand_kg"] = np.round(base_demand).astype(int)

# Set demand depot (id == 0) menjadi 0
df.loc[df["id"] == 0, "demand_kg"] = 0

# Simpan ke demand_kg.csv
df[["id", "nama", "kecamatan", "demand_kg"]].to_csv("data/semarang/demand_kg.csv", index=False)
print("File demand_kg.csv berhasil dibuat!")

File demand_kg.csv berhasil dibuat!


In [11]:
import requests
import pandas as pd
import numpy as np

df_titik = pd.read_csv("data/semarang/titik.csv").sort_values("id")

# Susun string koordinat dengan urutan wajib: lon,lat dipisahkan titik koma
coords_str = ";".join([f"{row['lon']},{row['lat']}" for _, row in df_titik.iterrows()])

# Panggil OSRM Table API
url = f"http://router.project-osrm.org/table/v1/driving/{coords_str}?annotations=distance,duration"
response = requests.get(url)
data = response.json()

if data.get("code") == "Ok":
    dist_matrix = np.array(data["distances"])  # satuan meter
    dur_matrix = np.array(data["durations"])   # satuan detik

    pd.DataFrame(dist_matrix).to_csv("data/semarang/jarak_m.csv", index=False, header=False)
    pd.DataFrame(dur_matrix).to_csv("data/semarang/waktu_s.csv", index=False, header=False)
    print("Matriks jarak_m.csv dan waktu_s.csv berhasil disimpan!")
else:
    print("Gagal mengambil dari OSRM:", data)

Matriks jarak_m.csv dan waktu_s.csv berhasil disimpan!


In [12]:
import numpy as np
import pandas as pd

def load_semarang_instance(kapasitas_truk_kg=3000):
    df_titik = pd.read_csv("data/semarang/titik.csv").sort_values("id")
    df_demand = pd.read_csv("data/semarang/demand_kg.csv").sort_values("id")
    jarak_m = pd.read_csv("data/semarang/jarak_m.csv", header=None).values

    inst = {
        "name": "Semarang-Case-Study",
        "node_coord": df_titik[["lon", "lat"]].values,
        "demand": df_demand["demand_kg"].values,
        "capacity": kapasitas_truk_kg,
        "depot": np.array([0]),
        "edge_weight": jarak_m
    }
    return inst

# --- CARA PENGGUNAAN ---
# 1. Panggil fungsi (misal: kapasitas truk ringan/sedang 3000 kg)
data_semarang = load_semarang_instance(kapasitas_truk_kg=3000)

# 2. Akses komponen datanya sama persis seperti saat mengakses data CVRPLIB:
koordinat = data_semarang["node_coord"]
permintaan = data_semarang["demand"]
kapasitas_Q = data_semarang["capacity"]
matriks_jarak = data_semarang["edge_weight"]

print("Nama Instans:", data_semarang["name"])
print("Total Node (1 Depot + Pelanggan):", len(permintaan))
print("Kapasitas Kendaraan (Q):", kapasitas_Q, "kg")
print("Ukuran Matriks Jarak:", matriks_jarak.shape)

Nama Instans: Semarang-Case-Study
Total Node (1 Depot + Pelanggan): 31
Kapasitas Kendaraan (Q): 3000 kg
Ukuran Matriks Jarak: (31, 31)
